# 06 · Decision trees, random forests and gradient boosting

After this notebook you can compute Gini and entropy by hand, find a tree's best split from scratch, explain why deep trees overfit, why a random forest fixes that (bagging, out-of-bag score, feature importance done right), and how gradient boosting builds a model by fitting residuals. You will also know when to reach for XGBoost, LightGBM or CatBoost.

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min · **Needs:** [05 · Overfitting and cross-validation](05_overfitting_regularization_cross_validation.ipynb)

## Why this matters in interviews

- Gradient-boosted trees are still the default for **tabular** data (churn, fraud, pricing, ranking features). "Why not just use an LLM for this?" has a crisp answer, and it lives here.
- Trees are the classic whiteboard algorithm: "how does a tree choose a split?", "why does a random forest beat a single tree?", "bagging vs boosting?" come up constantly.
- Feature importance is a trap question. Impurity-based importance is biased, and knowing about permutation importance marks you as someone who has shipped models.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `fo27` | When should you NOT use an LLM? |
| `fo11` | What is the difference between AI, machine learning, deep learning and generative AI? |

**Also asked in classical-ML rounds:** Gini vs entropy; how is a split chosen and what is information gain? why do deep trees overfit and how do you stop it? bagging vs boosting; what is the out-of-bag score? why are impurity importances biased? XGBoost vs LightGBM vs CatBoost; do trees need feature scaling?

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.datasets import load_breast_cancer, load_iris, make_classification, make_moons
from sklearn.ensemble import (GradientBoostingClassifier, GradientBoostingRegressor, HistGradientBoostingClassifier,
                              RandomForestClassifier)
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text, plot_tree

rng = np.random.default_rng(0)
BLUE, ORANGE, AQUA, YELLOW, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a8984"
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, AQUA, YELLOW, "#4a3aa7"])})
BLUE_ORANGE = LinearSegmentedColormap.from_list("blue_orange", [BLUE, "#f0efec", ORANGE])

def plot_boundary(ax, model, X, y, title):
    """Shade P(class 1) on a grid, draw the 0.5 boundary, then the points."""
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 150),
                         np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 150))
    p = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, p, levels=np.linspace(0, 1, 11), cmap=BLUE_ORANGE, alpha=0.45)
    ax.contour(xx, yy, p, levels=[0.5], colors="black", linewidths=1.2)
    ax.scatter(X[:, 0], X[:, 1], c=np.where(y == 1, ORANGE, BLUE), s=10, edgecolors="white", linewidths=0.3)
    ax.set(title=title, xlabel="x1", ylabel="x2")

## 1. Impurity: how mixed is a node?

**In plain English:** a decision tree asks yes/no questions ("age < 40?") and wants each answer group to be as *pure* as possible, ideally one class only. Impurity measures how mixed a group is.

For class proportions $p_k$ in a node:

$$\text{Gini} = 1 - \sum_k p_k^2 \qquad \text{Entropy} = -\sum_k p_k \log_2 p_k$$

Gini is the chance that two random picks from the node disagree; entropy is the bits needed to encode the label. Both are 0 for a pure node and maximal at 50/50. In practice they pick nearly the same splits; Gini is slightly cheaper (no logarithm) and is scikit-learn's default.

In [ ]:
def gini(labels):
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    return 1 - np.sum(p ** 2)

def entropy(labels):
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    return -np.sum(p * np.log2(p))

assert gini([0, 0, 0, 0]) == 0 and entropy([1, 1, 1]) == 0            # pure node
assert np.isclose(gini([0, 1]), 0.5) and np.isclose(entropy([0, 1]), 1.0)  # 50/50 is the worst case

p = np.linspace(0.001, 0.999, 300)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(p, 1 - p ** 2 - (1 - p) ** 2, lw=2, label="Gini")
ax.plot(p, -(p * np.log2(p) + (1 - p) * np.log2(1 - p)), lw=2, label="entropy (bits)")
ax.plot(p, 1 - np.maximum(p, 1 - p), lw=2, ls="--", label="misclassification rate")
ax.set(title="Impurity of a two-class node", xlabel="share of class 1 in the node", ylabel="impurity"); ax.legend()
plt.show()

## 2. Finding the best split from scratch

For every feature and every threshold (midpoints between sorted values), split the node in two and compute the **weighted impurity of the children**. The best split is the one with the biggest drop from the parent, the **information gain** (called impurity decrease in scikit-learn). A tiny churn table:

In [ ]:
toy = pd.DataFrame({
    "age":             [22, 25, 28, 33, 38, 41, 47, 52, 58, 63, 67, 71],
    "monthly_charges": [95, 88, 45, 92, 40, 81, 38, 99, 42, 55, 35, 48],
    "churned":         [1,  1,  0,  1,  0,  1,  0,  1,  0,  0,  0,  0],
})

def best_split(X, y):
    """Exhaustive search over features and midpoint thresholds, minimising weighted Gini."""
    parent, rows = gini(y), []
    for j in range(X.shape[1]):
        values = np.unique(X[:, j])
        for t in (values[:-1] + values[1:]) / 2:
            left, right = y[X[:, j] <= t], y[X[:, j] > t]
            child = (len(left) * gini(left) + len(right) * gini(right)) / len(y)
            rows.append((j, t, child, parent - child))
    return pd.DataFrame(rows, columns=["feature", "threshold", "weighted_child_gini", "gain"]).sort_values("gain", ascending=False)

X_toy, y_toy = toy[["age", "monthly_charges"]].to_numpy(float), toy["churned"].to_numpy()
splits = best_split(X_toy, y_toy)
splits["feature"] = splits["feature"].map({0: "age", 1: "monthly_charges"})
print(f"parent Gini: {gini(y_toy):.3f}")
print(splits.head(4).round(3).to_string(index=False))

In [ ]:
stump = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_toy, y_toy)
sk_feature, sk_threshold = ["age", "monthly_charges"][stump.tree_.feature[0]], stump.tree_.threshold[0]
print(f"scikit-learn's root split: {sk_feature} <= {sk_threshold}")
best = splits.iloc[0]
assert best["feature"] == sk_feature and np.isclose(best["threshold"], sk_threshold)
assert best["gain"] > splits.iloc[1]["gain"]                       # a unique winner, no tie

Our exhaustive search and scikit-learn agree on the first question. A full tree just repeats this on each child until the nodes are pure or a stopping rule fires (`max_depth`, `min_samples_leaf`, `min_impurity_decrease`). It is **greedy**: each split is locally best, with no look-ahead. Because only the *order* of a feature's values matters, trees need **no feature scaling** and shrug off monotone transformations like log.

## 3. Depth is capacity: deep trees overfit

In [ ]:
Xm, ym = make_moons(n_samples=600, noise=0.35, random_state=1)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.4, random_state=0)
depths = range(1, 16)
tr_acc, te_acc = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xm_tr, ym_tr)
    tr_acc.append(t.score(Xm_tr, ym_tr)); te_acc.append(t.score(Xm_te, ym_te))
best_depth = list(depths)[int(np.argmax(te_acc))]

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(depths, tr_acc, "o-", lw=2, label="train accuracy")
ax.plot(depths, te_acc, "o-", lw=2, label="test accuracy")
ax.axvline(best_depth, color=GREY, ls="--", label=f"best depth = {best_depth}")
ax.set(title="Decision tree on noisy moons: depth vs accuracy", xlabel="max_depth", ylabel="accuracy"); ax.legend()
plt.show()
print(f"depth {best_depth}: test {max(te_acc):.3f}   depth 15: train {tr_acc[-1]:.3f}, test {te_acc[-1]:.3f}")
assert tr_acc[-1] > 0.99 and te_acc[-1] < max(te_acc) - 0.03

An unlimited tree keeps splitting until every leaf is pure, so it memorises every noisy point: train accuracy near 100%, test accuracy falling. The knobs that limit it: `max_depth`, `min_samples_leaf`, `max_leaf_nodes` and cost-complexity pruning (`ccp_alpha`).

### What a small tree looks like

A depth-2 tree on iris is a flowchart you can read aloud. That readability is the single tree's big advantage.

In [ ]:
iris = load_iris()
small = DecisionTreeClassifier(max_depth=2, random_state=0).fit(iris.data, iris.target)
fig, ax = plt.subplots(figsize=(10, 4.2))
plot_tree(small, feature_names=iris.feature_names, class_names=list(iris.target_names), filled=True,
          rounded=True, fontsize=9, ax=ax)
ax.set_title("A depth-2 decision tree on iris"); plt.show()
print(export_text(small, feature_names=list(iris.feature_names)))

## 4. Bagging and random forests

**In plain English:** one deep tree is a nervous expert: brilliant on what it has seen, jumpy on anything new (high variance). Train hundreds of them on slightly different data and average their votes, and the jumpiness cancels out.

- **Bagging** (bootstrap aggregating): each tree trains on a bootstrap sample (n rows drawn *with replacement*, so about 63% are distinct).
- **Random forest** = bagging + a random subset of features at each split (`max_features="sqrt"` for classification). That de-correlates the trees, which is what makes averaging effective.
- **Out-of-bag (OOB) score:** each row is left out of about 37% of the trees, so those trees can score it. It is a free validation estimate, no hold-out set needed.

In [ ]:
bc = load_breast_cancer()
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(bc.data, bc.target, test_size=0.3, stratify=bc.target, random_state=0)
tree = DecisionTreeClassifier(random_state=0).fit(Xb_tr, yb_tr)
forest = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=0).fit(Xb_tr, yb_tr)
print(f"single tree  : test accuracy {tree.score(Xb_te, yb_te):.3f}")
print(f"random forest: test accuracy {forest.score(Xb_te, yb_te):.3f}   OOB estimate {forest.oob_score_:.3f}")
print(f"share of distinct rows in one bootstrap sample: {len(np.unique(rng.integers(0, 1000, 1000))) / 1000:.3f}  (theory 1 - 1/e = 0.632)")
assert forest.score(Xb_te, yb_te) >= tree.score(Xb_te, yb_te) and abs(forest.oob_score_ - forest.score(Xb_te, yb_te)) < 0.05

In [ ]:
sizes = [1, 5, 10, 25, 50, 100, 200]
accs = [RandomForestClassifier(n_estimators=k, random_state=0).fit(Xb_tr, yb_tr).score(Xb_te, yb_te) for k in sizes]
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(sizes, accs, "o-", lw=2, label="random forest")
ax.axhline(tree.score(Xb_te, yb_te), color=GREY, ls="--", label="single deep tree")
ax.set(xscale="log", title="More trees: better, then flat (never worse)", xlabel="number of trees", ylabel="test accuracy")
ax.legend(); plt.show()

Adding trees never causes overfitting in the "more capacity" sense; it only costs time, and the curve flattens after a few hundred trees.

### Feature importance: impurity-based vs permutation

`feature_importances_` sums each feature's impurity decrease over all splits. It is fast, but it is computed on *training* data and favours features with many distinct values: a random ID column gets credit for memorising noise. **Permutation importance** shuffles one column on held-out data and measures how much the score drops: slower, but honest. A noisy synthetic problem with 3 real signals, 3 useless features and one column of random numbers:

In [ ]:
Xf, yf = make_classification(n_samples=1000, n_features=6, n_informative=3, n_redundant=0, flip_y=0.1,
                             shuffle=False, random_state=0)            # f0-f2 informative, f3-f5 useless
Xf = np.c_[Xf, np.random.default_rng(5).random(len(yf))]                # + a random, high-cardinality column
names = ["f0 (signal)", "f1 (signal)", "f2 (signal)", "f3 (useless)", "f4 (useless)", "f5 (useless)", "RANDOM NOISE"]
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(Xf, yf, test_size=0.3, random_state=0)
rf_n = RandomForestClassifier(n_estimators=200, random_state=0).fit(Xf_tr, yf_tr)
imp = pd.Series(rf_n.feature_importances_, index=names)
perm_s = pd.Series(permutation_importance(rf_n, Xf_te, yf_te, n_repeats=5, random_state=0).importances_mean, index=names)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
colors = [ORANGE if n == "RANDOM NOISE" else BLUE for n in names[::-1]]
axes[0].barh(names[::-1], imp[names[::-1]], color=colors); axes[0].set(title="Impurity-based (training data)", xlabel="importance")
axes[1].barh(names[::-1], perm_s[names[::-1]], color=colors); axes[1].set(title="Permutation (test data)", xlabel="accuracy drop when shuffled")
axes[1].axvline(0, color=GREY, lw=0.8); plt.tight_layout(); plt.show()
print(f"RANDOM NOISE: impurity importance {imp['RANDOM NOISE']:.3f}, permutation importance {perm_s['RANDOM NOISE']:+.3f}")
print(f"useless f3-f5: impurity {imp.iloc[3:6].round(3).tolist()}, permutation {perm_s.iloc[3:6].round(3).tolist()}")
assert imp["RANDOM NOISE"] > 0.03 and abs(perm_s["RANDOM NOISE"]) < 0.01

The random column earns as much impurity importance as the useless real features, because deep trees use it to split off leftover noisy training points. On test data, shuffling any of them changes little or nothing (a percentage point or two at most, within noise), while shuffling a real signal costs far more, and permutation importance says so. Two more caveats worth saying: with **correlated features** (radius, perimeter and area in breast cancer) both methods split the credit or hide it, because the model can use either copy; and importance says what the *model* relies on, not what *causes* the outcome.

## 5. Gradient boosting: fit the residuals, again and again

**In plain English:** start with a dumb guess (the mean). Look at what you got wrong, the residuals. Fit a small tree to those mistakes, add a *fraction* of it (the learning rate) to your model, and repeat. Each tree fixes what the previous ones missed.

For squared error, the residual $y - F(x)$ is exactly the negative gradient of the loss with respect to the prediction. That is why it is called *gradient* boosting: it is gradient descent in function space, with a tree as each step. Five rounds with stumps (depth-1 trees), from scratch:

In [ ]:
xr = np.sort(rng.uniform(0, 6, 80))
yr = np.sin(xr) + 0.3 * xr + rng.normal(0, 0.25, 80)
Xr = xr[:, None]
lr_boost, n_rounds = 0.5, 5

F = np.full_like(yr, yr.mean())                  # round 0: predict the mean
stumps, train_mse = [], [np.mean((yr - F) ** 2)]
for m in range(n_rounds):
    residual = yr - F                            # = negative gradient of squared error
    stump_m = DecisionTreeRegressor(max_depth=1).fit(Xr, residual)
    F = F + lr_boost * stump_m.predict(Xr)
    stumps.append(stump_m); train_mse.append(np.mean((yr - F) ** 2))
print("training MSE by round:", np.round(train_mse, 4))
assert all(a > b for a, b in zip(train_mse, train_mse[1:]))      # every round helps on training data

sk_gb = GradientBoostingRegressor(n_estimators=n_rounds, learning_rate=lr_boost, max_depth=1).fit(Xr, yr)
assert np.allclose(sk_gb.predict(Xr), F)                          # identical to scikit-learn's algorithm
print("our 5-round model == GradientBoostingRegressor(n_estimators=5, max_depth=1, learning_rate=0.5)")

In [ ]:
def boosted_predict(xs, rounds):
    out = np.full(len(xs), yr.mean())
    for s in stumps[:rounds]:
        out += lr_boost * s.predict(xs[:, None])
    return out

grid = np.linspace(0, 6, 400)
fig, axes = plt.subplots(1, 4, figsize=(11, 3.2), sharey=True)
for ax, r in zip(axes, [0, 1, 3, 5]):
    ax.scatter(xr, yr, s=10, color=BLUE, alpha=0.7)
    ax.plot(grid, boosted_predict(grid, r), color=ORANGE, lw=2)
    ax.set(title=f"after {r} round{'s' if r != 1 else ''}", xlabel="x")
axes[0].set_ylabel("y"); fig.suptitle("Gradient boosting with stumps: each round adds one step to the staircase", y=1.03)
plt.tight_layout(); plt.show()

| | Bagging / random forest | Boosting |
|---|---|---|
| Trees are built | independently, in parallel | one after another, each fixing the last |
| Each tree is | deep (low bias, high variance) | shallow (high bias, low variance) |
| Reduces mainly | variance | bias |
| Too many trees | harmless | overfits: use early stopping |
| Key knobs | `n_estimators`, `max_features` | `learning_rate` × `n_estimators`, depth, subsampling |

## 6. `HistGradientBoostingClassifier`: the modern default in scikit-learn

It bins each feature into at most 255 buckets first, so finding a split means scanning a histogram instead of sorting (the LightGBM trick). It is much faster on large data (the gap grows with the number of rows), handles missing values natively, and supports categorical features and early stopping (`early_stopping="auto"` switches it on above 10,000 rows).

In [ ]:
Xh, yh = make_classification(n_samples=6000, n_features=20, n_informative=8, flip_y=0.03, random_state=0)
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, yh, test_size=0.25, random_state=0)
Xh_tr[rng.random(Xh_tr.shape) < 0.05] = np.nan               # 5% missing values: no imputation needed
Xh_te[rng.random(Xh_te.shape) < 0.05] = np.nan

rows = []
for name, est in [("HistGradientBoosting", HistGradientBoostingClassifier(random_state=0)),
                  ("GradientBoosting (classic)", GradientBoostingClassifier(n_estimators=100, random_state=0))]:
    Xa, Xb_ = (Xh_tr, Xh_te) if name.startswith("Hist") else (np.nan_to_num(Xh_tr), np.nan_to_num(Xh_te))
    t0 = time.perf_counter()
    est.fit(Xa, yh_tr)
    rows.append((name, time.perf_counter() - t0, est.score(Xb_, yh_te)))
for name, secs, acc in rows:
    print(f"{name:>27}: fit {secs:5.2f}s   test accuracy {acc:.3f}")
hgb = HistGradientBoostingClassifier(max_iter=500, early_stopping=True, random_state=0).fit(Xh_tr, yh_tr)
print(f"with early_stopping=True (10% of train held out, patience 10): used {hgb.n_iter_} of max_iter=500 rounds")
assert rows[0][2] > 0.9 and rows[0][2] >= rows[1][2] - 0.01 and hgb.n_iter_ < 500

### XGBoost, LightGBM, CatBoost

Not installed here (and not needed to learn the ideas), but you will be asked to compare them. All three are gradient-boosted trees with regularised objectives and second-order (Newton) steps.

| | XGBoost | LightGBM | CatBoost |
|---|---|---|---|
| Tree growth | level-wise (depth by depth) by default | **leaf-wise** (split the best leaf anywhere): faster, can overfit small data | **symmetric** (oblivious) trees: same split across a level, fast inference |
| Split finding | exact or histogram | histogram + GOSS sampling + feature bundling | histogram |
| Categorical features | one-hot or native (newer versions) | native | **best-in-class**: ordered target statistics, no leakage |
| Strength | mature, everywhere, strong on Kaggle | speed on large data | great defaults, little tuning, categorical-heavy data |
| scikit-learn analogue | `GradientBoostingClassifier` | `HistGradientBoostingClassifier` | (none) |

Their scikit-learn-style APIs look the same (needs `pip install xgboost`):

```python
from xgboost import XGBClassifier
model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8,
                      colsample_bytree=0.8, early_stopping_rounds=50, eval_metric="logloss")
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
```

## 7. Decision boundaries: tree vs forest vs boosting

In [ ]:
models = {
    "single tree (no depth limit)": DecisionTreeClassifier(random_state=0),
    "random forest (200 trees)": RandomForestClassifier(n_estimators=200, random_state=0),
    "gradient boosting (HistGB)": HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=0),
}
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8), sharey=True)
scores = {}
for ax, (name, m) in zip(axes, models.items()):
    m.fit(Xm_tr, ym_tr)
    scores[name] = m.score(Xm_te, ym_te)
    plot_boundary(ax, m, Xm, ym, f"{name}\ntest accuracy {scores[name]:.3f}")
    if ax is not axes[0]:
        ax.set_ylabel("")
plt.tight_layout(); plt.show()
assert scores["random forest (200 trees)"] > scores["single tree (no depth limit)"]

All three draw boundaries out of axis-aligned rectangles. The single tree carves out islands around individual noisy points (overfitting you can see). The forest averages 200 such trees, so its probabilities are graded (the pale bands) and it generalises better. Boosting with 200 rounds and no early stopping is also carving islands in the overlap: on data this noisy it needs fewer rounds, a smaller learning rate or early stopping (exercise 3). Another thing to notice: outside the data, tree models predict a **constant**. They cannot extrapolate a trend, which matters for forecasting prices or anything that grows over time.

**Why this beats an LLM for tabular prediction (`fo27`, `fo11`):** a boosted-tree model trains in seconds on a CPU, predicts in microseconds, is deterministic and auditable, and on structured data with clear features it is usually more accurate than prompting a language model with the row as text. Use the LLM for the messy-language parts (extracting the features from an email) and the trees for the prediction.

## Try it yourself

**1.** Rerun the best-split search using **entropy** instead of Gini, and check it against `DecisionTreeClassifier(criterion="entropy", max_depth=1)`.

In [ ]:
# Solution — try it yourself first, then run this
def best_split_entropy(X, y):
    best = (None, None, -1.0)
    for j in range(X.shape[1]):
        values = np.unique(X[:, j])
        for t in (values[:-1] + values[1:]) / 2:
            left, right = y[X[:, j] <= t], y[X[:, j] > t]
            gain = entropy(y) - (len(left) * entropy(left) + len(right) * entropy(right)) / len(y)
            if gain > best[2]:
                best = (j, t, gain)
    return best

j, t, gain = best_split_entropy(X_toy, y_toy)
sk_e = DecisionTreeClassifier(criterion="entropy", max_depth=1, random_state=0).fit(X_toy, y_toy)
print(f"ours: feature {j}, threshold {t}, information gain {gain:.3f} bits;  sklearn: feature {sk_e.tree_.feature[0]}, threshold {sk_e.tree_.threshold[0]}")
assert j == sk_e.tree_.feature[0] and np.isclose(t, sk_e.tree_.threshold[0])

**2.** Regularise the single tree instead of averaging: use 5-fold cross-validation on the moons training set to choose `min_samples_leaf` from `[1, 2, 5, 10, 20, 40]`. How close does the pruned tree get to the forest?

In [ ]:
# Solution — try it yourself first, then run this
cv_scores = {k: cross_val_score(DecisionTreeClassifier(min_samples_leaf=k, random_state=0), Xm_tr, ym_tr, cv=5).mean()
             for k in [1, 2, 5, 10, 20, 40]}
best_leaf = max(cv_scores, key=cv_scores.get)
pruned = DecisionTreeClassifier(min_samples_leaf=best_leaf, random_state=0).fit(Xm_tr, ym_tr)
print({k: round(float(v), 3) for k, v in cv_scores.items()})
print(f"min_samples_leaf={best_leaf}: test {pruned.score(Xm_te, ym_te):.3f}   vs forest {scores['random forest (200 trees)']:.3f}")

**3.** Learning rate × rounds. Train `GradientBoostingClassifier` on the moons data with `learning_rate=1.0` and `0.1` (300 rounds, depth 3). Use `staged_predict` to plot test accuracy per round. Which one peaks early and then degrades?

In [ ]:
# Solution — try it yourself first, then run this
fig, ax = plt.subplots(figsize=(8, 3.2))
for lr_ in (1.0, 0.1):
    gbc = GradientBoostingClassifier(n_estimators=300, learning_rate=lr_, max_depth=3, random_state=0).fit(Xm_tr, ym_tr)
    staged = [np.mean(p == ym_te) for p in gbc.staged_predict(Xm_te)]
    ax.plot(range(1, 301), staged, lw=2, label=f"learning_rate={lr_}: best {max(staged):.3f} at round {int(np.argmax(staged)) + 1}")
ax.set(title="Smaller steps need more rounds but overfit more gently", xlabel="round", ylabel="test accuracy"); ax.legend()
plt.show()

## Say it in an interview

- **30-second answer:** "A decision tree greedily picks the feature and threshold that most reduce impurity (Gini or entropy) and recurses. Deep trees have low bias but high variance, so they overfit. A random forest averages many deep trees trained on bootstrap samples with random feature subsets, which cuts variance, and gives a free out-of-bag estimate. Gradient boosting instead adds shallow trees one at a time, each fitted to the current residuals (the negative gradient), scaled by a learning rate. It cuts bias and needs early stopping."
- **Numbers:** a bootstrap sample holds about 63% distinct rows (so about 37% are out-of-bag); Gini at 50/50 is 0.5 and entropy is 1 bit; histogram boosting uses at most 255 bins per feature.
- **When to use:** tabular data with mixed feature types and some missing values: start with `HistGradientBoostingClassifier` or LightGBM/XGBoost; a random forest when you want strong results with almost no tuning; a single shallow tree when a human must read the rules.
- **Traps:** trusting impurity importance (biased to high-cardinality and training-set noise; use permutation importance on held-out data); expecting trees to extrapolate; one-hot encoding a 10,000-level categorical for a tree (use native categorical support or target encoding done inside CV); scaling features for trees (unnecessary); boosting without early stopping.
- **The GenAI angle (`fo27`):** do not send a table row to an LLM to predict churn. Use the LLM to turn messy text into features, and gradient-boosted trees to predict.

## Next

- [07 · kNN, SVM and Naive Bayes](07_knn_svm_naive_bayes.ipynb): three more classic classifiers and when each wins.
- [09 · Feature engineering and pipelines](09_feature_engineering_and_pipelines.ipynb): encoding categoricals for trees and linear models.
- [ML model end to end (coding round)](../17_interview_coding_rounds/06_ml_model_end_to_end.ipynb).
- Theory: [ml_fundamentals course](../../ml_fundamentals/index.html) (chapter 9 and the gradient-boosting tools page), [interview bank](../../ai_interview_prep/index.html).